In [4]:
import pandas as pd

accidents = pd.read_csv("../data/raw/accident_numbers.csv")
cities = pd.read_csv("../data/raw/large_cities_fatalities.csv")
states = pd.read_csv("../data/raw/state_wise_accidents.csv")
violations = pd.read_csv("../data/raw/type_of_violation.csv")

In [5]:
# 1) Accident 

num_cols = ["Accidents", "Fatalities", "Persons Injured"]

for col in num_cols:
    accidents[col] = (      # overwriting the column
        accidents[col]    
        .astype(str)    # convert to string
        .str.replace(",", "", regex=True)    # remove commas
        .astype(int)     # convert back to int
    )

accidents["fatality_rate"] = (accidents["Fatalities"] / accidents["Accidents"] ) * 100
accidents["fatality_rate"] = accidents["fatality_rate"].round(2)

accidents = accidents.rename(columns={
    "Year": "year",
    "Accidents": "accidents",
    "% Change from previous year": "accidents_change_percent",
    "Fatalities": "fatalities",
    "% Change from previous year.1": "fatalities_change_percent",
    "Persons Injured": "persons_injured",
    "% Change from previous year.2": "persons_injured_change_percent"
})

accidents.to_csv("../data/cleaned/accidents_cleaned.csv", index=False)

accidents.head()


,year,accidents,accidents_change_percent,fatalities,fatalities_change_percent,persons_injured,persons_injured_change_percent,fatality_rate
0,2019,456959,-2.9,158984,0.9,449360,-3.3,34.79
1,2020,372181,-18.6,138383,-13.0,346747,-22.8,37.18
2,2021,412432,10.8,153972,11.3,384448,10.9,37.33
3,2022,461312,11.9,168491,9.4,443366,15.3,36.52
4,2023,480583,4.2,172890,2.6,462825,4.4,35.98


In [6]:
# 2) Cities

cities = cities.rename(columns={
    "City": "city",
    "Overspeeding": "overspeeding",
    "Drunken Driving/Drugs use": "drunken_driving_drugs_use",
    "Wrong side driving": "wrong_side_driving",
    "Jumping red light": "jumping_red_light",
    "Using Mobile Phone": "using_mobile_phone",
    "Others": "others"
})

# removes rows where City is NaN and rows where City contains "Total" (unwanted)
cities = cities[cities["city"].notna()]
cities = cities[~cities["city"].str.contains("Total", case=False)]

# convert all columns except "City" to numeric and missing/invalid values to NaN
violation_cols = cities.columns.drop("city")
cities[violation_cols] = cities[violation_cols].apply(pd.to_numeric, errors="coerce")

# total fatalities per city
cities["total_fatalities"] = cities[violation_cols].sum(axis=1)

cities.to_csv("../data/cleaned/city_violation_fatalities.csv", index=False)

cities.head()

,Sl No,city,overspeeding,drunken_driving_drugs_use,wrong_side_driving,jumping_red_light,using_mobile_phone,others,total_fatalities
0,1.0,Agra,234,8,47,11,4,195,500.0
1,2.0,Ahmedabad,515,2,7,11,0,0,537.0
2,3.0,Allahabad (Prayagraj),182,56,139,61,93,51,585.0
3,4.0,Amritsar,53,2,10,0,0,45,114.0
4,5.0,Asansol Durgapur,149,0,1,0,0,203,358.0


In [7]:
# 3) States

# removes rows where State is NaN and rows where State contains "#" or "Total" (unwanted)
states = states[~states["State"].str.contains("#", na=False)]
states = states[~states["State"].str.contains("Total", na=False)]

# convert year columns to numeric and missing/invalid values to NaN
year_cols = [
    "2019 Accidents",
    "2020 Accidents",
    "2021 Accidents",
    "2022 Accidents",
    "2023 Accidents"
]
for col in year_cols:
    states[col] = pd.to_numeric(states[col], errors="coerce")

# reshaping the dataframe from wide to long format for easier analysis 
state_long = states.melt(id_vars=["State"], value_vars=year_cols, var_name="Year", value_name="Accidents")

# extracting the year from the column names and converting it to integer
state_long["Year"] = (state_long["Year"].str.extract("(\d{4})").astype(int))

# removing rows where Accidents col has missing values
state_long = state_long.dropna(subset=["Accidents"])

state_long = state_long.rename(columns={
    "State": "state",
    "Year": "year",
    "Accidents": "accidents"
})

state_long.to_csv("../data/cleaned/state_accidents_long.csv", index=False)

state_long.head()

<>:22: SyntaxWarning: invalid escape sequence '\d'
<>:22: SyntaxWarning: invalid escape sequence '\d'
/var/folders/k6/q3_zm6pj0h7gwwqjdk1_t6cm0000gn/T/ipykernel_39218/2981325715.py:22: SyntaxWarning: invalid escape sequence '\d'
  state_long["Year"] = (state_long["Year"].str.extract("(\d{4})").astype(int))


,state,year,accidents
0,Andhra Pradesh,2019,21992.0
1,Arunachal Pradesh,2019,237.0
2,Assam,2019,8350.0
3,Bihar,2019,10007.0
4,Chhattisgarh,2019,13899.0


In [8]:
# 4) Violations

violations = violations.rename(columns={
    "Category": "category",
    "2022-Accidents": "accidents_2022",
    "2022-Killed": "killed_2022",
    "2022-injured": "injured_2022",
    "2023-Accidents": "accidents_2023",
    "2023-Killed": "killed_2023",
    "2023-injured": "injured_2023",
    "%Change-Accidents": "accidents_change_percent",
    "%Change-killed": "killed_change_percent",
    "%Change-Injured": "injured_change_percent",
    "fatality_rate_2022": "fatality_rate_2022",
    "fatality_rate_2023": "fatality_rate_2023"
})

# removes rows where Category is NaN and rows where "Category" contains "%" sign
violations = violations[~violations["category"].str.contains("%", na=False)]

# removes rows where Category contains "All India" (unwanted total data)
violations = violations[~violations["category"].str.contains("All India", na=False)]

# convert year columns to numeric and missing/invalid values to NaN
num_cols = [
    "accidents_2022", "killed_2022", "injured_2022",
    "accidents_2023", "killed_2023", "injured_2023"
]
for col in num_cols:
    violations[col] = (
        violations[col]
        .astype(str)
        .str.replace(",", "", regex=True)
        .astype(int)
    )

violations["fatality_rate_2022"] = ((violations["killed_2022"] / violations["accidents_2022"]) * 100).round(2)
violations["fatality_rate_2023"] = ((violations["killed_2023"] / violations["accidents_2023"]) * 100).round(2)

violations.to_csv("../data/cleaned/violation_severity_clean.csv", index=False)

violations.head()

,category,accidents_2022,killed_2022,injured_2022,accidents_2023,killed_2023,injured_2023,accidents_change_percent,killed_change_percent,injured_change_percent,fatality_rate_2022,fatality_rate_2023
0,Over-speeding,333323,119904,322795,328727,117682,320416,-1.38,-1.85,-0.74,35.97,35.80
2,Drunken driving/consumption of alcohol & drug,10080,4201,8809,9143,3674,8421,-9.30,-12.54,-4.40,41.68,40.18
4,Driving on wrong side/Lane indiscipline,22586,9094,21745,25242,9432,24435,11.76,3.72,12.37,40.26,37.37
6,Jumping red light,4021,1462,3450,2440,818,2157,-39.32,-44.05,-37.48,36.36,33.52
8,Use of mobile phone,7558,3395,6255,7122,2884,6445,-5.77,-15.05,3.04,44.92,40.49
